# Bài 06 — Range, Null Space và Rank với NumPy

Tài liệu đi kèm gồm [lý thuyết](notes.md) và [ứng dụng ML](ml-in-practice.md).

## Mục tiêu

- Xác định shape và số chiều của bốn không gian con của một ma trận.
- Kiểm chứng basis tìm tay và lấy basis bằng SVD đầy đủ.
- Phân loại nghiệm của $Ax=b$ bằng phần dư và nullity.
- Thay đổi tolerance và giải thích numerical rank từ các singular value.
- Dịch trọng số hồi quy theo kernel rồi so sánh dự đoán trên train và dữ liệu mới.
- Tái tạo bảng điểm phim hạng thấp và fit một người dùng mới từ các điểm đã biết.

Trước mỗi ví dụ, hãy dự đoán shape, rank hoặc phần dư rồi mới chạy ô code.

## 0. Khởi tạo

Notebook dùng NumPy. Seed 42 giúp lặp lại ví dụ có nhiễu.

In [1]:
import numpy as np

ATOL = 1e-12

## 1. Range, row space và kernel từ các cột và hàng gốc

Với `A.shape == (3, 4)`, một vector đầu vào có 4 thành phần và một vector đầu ra
có 3 thành phần. Trừ hai hàng đầu khỏi hàng cuối để đọc hai pivot và hai biến tự do.

In [2]:
A = np.array([[1., 0., 1., 2.], [0., 1., 2., -1.], [1., 1., 3., 1.]])  # (3, 4)
C = A[:, :2]                                                        # (3, 2)
B = A[:2].T                                                         # (4, 2)
Z = np.array([[-1., -2.], [-2., 1.], [1., 0.], [0., 1.]])            # (4, 2)
L = np.array([[1.], [1.], [-1.]])                                    # (3, 1)

np.testing.assert_allclose(C @ [[1., 0., 1., 2.], [0., 1., 2., -1.]], A)
np.testing.assert_allclose(A @ Z, 0., atol=ATOL)
np.testing.assert_allclose(A.T @ L, 0., atol=ATOL)
np.testing.assert_allclose(B.T @ Z, 0., atol=ATOL)

r = np.linalg.matrix_rank(A)
assert np.linalg.matrix_rank(C) == np.linalg.matrix_rank(B) == r == 2
assert np.linalg.matrix_rank(Z) == A.shape[1] - r
assert np.linalg.matrix_rank(L) == A.shape[0] - r
print('Rank, nullity, left nullity:', r, Z.shape[1], L.shape[1])

Rank, nullity, left nullity: 2 2 1


## 2. Cùng đầu ra và nghiệm least squares

Cộng một vector kernel vào $x_0$ tạo một nghiệm khác của cùng hệ. Dùng `lstsq`
trên hai vế phải và kiểm tra $A^T(b-A\hat x)\approx 0$ cho cả hai. Khi có nhiều
nghiệm tối ưu, nghiệm do `lstsq` chọn vuông góc với kernel và có norm nhỏ nhất.

In [3]:
x0 = np.array([2., -1., 0., 0.])
x_shifted = x0 + Z @ np.array([2., -3.])
np.testing.assert_allclose(A @ x_shifted, A @ x0, atol=ATOL)

rhs = np.column_stack([A @ x0, [2., -1., 0.]])  # (3, 2)
solutions, _, _, _ = np.linalg.lstsq(A, rhs, rcond=None)
residual = rhs - A @ solutions
errors = np.linalg.norm(residual, axis=0)
np.testing.assert_allclose(A.T @ residual, 0., atol=ATOL)
np.testing.assert_allclose(Z.T @ solutions, 0., atol=ATOL)
np.testing.assert_allclose(errors, [0., 1. / np.sqrt(3.)], atol=ATOL)
print('Residual norm của hai đích:', errors)

Residual norm của hai đích: [6.28036983e-16 5.77350269e-01]


## 3. Đọc bốn basis từ SVD đầy đủ

SVD trả về `Vt` với mỗi hướng ở một **hàng**. Lấy `Vt[r:, :].T` để lưu các
vector kernel theo cột. So sánh phần dư và tính trực chuẩn với basis tìm tay ở phần 1.

In [4]:
left, s, right_t = np.linalg.svd(A, full_matrices=True)
column_basis = left[:, :r]                # (3, 2)
row_basis = right_t[:r, :].T               # (4, 2)
null_basis = right_t[r:, :].T              # (4, 2)
left_null_basis = left[:, r:]             # (3, 1)

np.testing.assert_allclose(A @ null_basis, 0., atol=ATOL)
np.testing.assert_allclose(A.T @ left_null_basis, 0., atol=ATOL)
for basis in (column_basis, row_basis, null_basis, left_null_basis):
    np.testing.assert_allclose(basis.T @ basis, np.eye(basis.shape[1]), atol=ATOL)
np.testing.assert_allclose(null_basis @ (null_basis.T @ Z), Z, atol=ATOL)

_, _, reduced_vt = np.linalg.svd(A, full_matrices=False)
print('Singular values:', s)
print('Shape Vt đầy đủ và rút gọn:', right_t.shape, reduced_vt.shape)
print('Số hướng kernel lấy được:', null_basis.shape[1], reduced_vt[r:].shape[0])
assert reduced_vt[r:].shape[0] == 1 < null_basis.shape[1] == 2

Singular values: [4.24264069e+00 2.44948974e+00 2.86658352e-16]
Shape Vt đầy đủ và rút gọn: (4, 4) (3, 4)
Số hướng kernel lấy được: 2 1


## 4. Đổi tolerance cho ma trận gần phụ thuộc

Hai singular value dương cho rank đại số 2. Hãy dự đoán khi nào numerical rank
xem singular value nhỏ hơn là dưới ngưỡng.

In [5]:
near = np.diag([1., 1e-9])
near_s = np.linalg.svd(near, compute_uv=False)
for tol, expected in [(1e-8, 1), (1e-10, 2)]:
    numerical_rank = np.linalg.matrix_rank(near, tol=tol)
    assert numerical_rank == np.count_nonzero(near_s > tol) == expected
    print(f'tol={tol:.0e}, rank={numerical_rank}, singular values={near_s}')

tol=1e-08, rank=1, singular values=[1.e+00 1.e-09]
tol=1e-10, rank=2, singular values=[1.e+00 1.e-09]


## 5. Feature phụ thuộc và trọng số không xác định duy nhất

Với hai cột feature `feature` và `2 * feature`, chỉ tổ hợp $w_1+2w_2$ ảnh hưởng
đến dự đoán. Cột đầu trong design matrix là intercept.

In [6]:
feature = np.array([-2., -1., 0., 1., 2.])
X = np.column_stack([np.ones_like(feature), feature, 2. * feature])
y = 1. + 3. * feature
w, _, design_rank, _ = np.linalg.lstsq(X, y, rcond=None)
feature_relation = np.array([0., 2., -1.])
w_alternative = w + 4. * feature_relation

assert design_rank == 2
np.testing.assert_allclose(X @ feature_relation, 0., atol=ATOL)
np.testing.assert_allclose(X @ w, y, atol=ATOL)
np.testing.assert_allclose(X @ w_alternative, y, atol=ATOL)
print('Hai bộ trọng số:', w, w_alternative)

Hai bộ trọng số: [1.  0.6 1.2] [ 1.   8.6 -2.8]


## 6. SVD cắt ngọn cho ma trận điểm phim

Mỗi hàng trong `ratings` là một người dùng và mỗi cột là một phim. Giữ $k$ hướng
để tái tạo bảng, rồi tính chuẩn Frobenius của hiệu bằng `ord='fro'`.

In [7]:
movies = np.array(['Hành động', 'Hoạt hình', 'Phiêu lưu', 'Trinh thám', 'Hài'])
user_profiles = np.array([[2., 0.], [0., 3.], [1., 1.], [2., 1.]])       # (4, 2)
movie_profiles = np.array([[2., 0., 1., 2., 1.], [0., 2., 1., 1., 2.]])  # (2, 5)
ratings = user_profiles @ movie_profiles                                # (4, 5)

rating_left, rating_s, rating_vt = np.linalg.svd(ratings, full_matrices=False)
for k in (1, 2):
    approximation = (rating_left[:, :k] * rating_s[:k]) @ rating_vt[:k]
    error = np.linalg.norm(ratings - approximation, ord='fro')
    np.testing.assert_allclose(error, np.linalg.norm(rating_s[k:]), atol=ATOL)
    print(f'k={k}, sai số tái tạo={error:.3g}')

assert np.linalg.matrix_rank(ratings) == 2
np.testing.assert_allclose(approximation, ratings, atol=ATOL)

k=1, sai số tái tạo=5.88
k=2, sai số tái tạo=5.73e-15


## Bài tập

Có 7 bài tăng dần và một bài tổng hợp. Hãy dự đoán shape và kết quả, rồi bỏ dấu
`#` và điền các dòng `TODO`. Chạy ô bài làm và ô kiểm tra ngay sau đó. Mở phần
đáp án khi cần đối chiếu. Các ô kiểm tra báo tên biến còn thiếu nếu bạn chạy
notebook trước khi điền bài.

In [8]:
def ready(*names):
    """Báo các biến TODO chưa được định nghĩa."""
    missing = [name for name in names if name not in globals()]
    if missing:
        print('Cần điền TODO cho ' + ', '.join(missing))
    return not missing

### Bài 1 — Đọc pivot và tìm basis bằng tay

Tìm basis cột từ các cột gốc theo thứ tự pivot và basis kernel từ các biến tự do của `ex1_A`. Mỗi vector basis nằm trên một cột. Tính rank bằng số pivot.

In [9]:
ex1_A = np.array([[1., 2., 0., 1.], [0., 0., 1., -1.], [2., 4., 1., 1.]])
# TODO: ex1_rank = ...
# TODO: ex1_col = ...       # (3, 2), lấy hai cột pivot của ex1_A
# TODO: ex1_null = ...      # (4, 2), cho từng biến tự do bằng 1

In [10]:
if ready('ex1_rank', 'ex1_col', 'ex1_null'):
    assert ex1_rank == 2
    assert ex1_col.shape == (3, ex1_rank) and ex1_null.shape == (4, 4 - ex1_rank)
    np.testing.assert_allclose(ex1_col, ex1_A[:, [0, 2]], atol=ATOL)
    assert np.linalg.matrix_rank(ex1_null) == 2
    np.testing.assert_allclose(ex1_A @ ex1_null, 0., atol=ATOL)
    print('Bài 1 đạt.')

Cần điền TODO cho ex1_rank, ex1_col, ex1_null


<details>
<summary>Đáp án Bài 1</summary>

```python
ex1_rank = 2
ex1_col = ex1_A[:, [0, 2]]
ex1_null = np.array([[-2., -1.], [1., 0.], [0., 1.], [0., 1.]])
```

Hàng cuối bằng hai lần hàng đầu cộng hàng thứ hai. Sau khi khử, hai biến tự do là $x_2$ và $x_4$.

</details>

### Bài 2 — Hai vế phải, hai kết quả

Giải cả hai hệ bằng một lần gọi `lstsq`. Tính $b-A\hat x$ và dùng norm theo cột để kiểm tra từng đích có thuộc range trong ngưỡng `ATOL` hay không.

In [11]:
ex2_A = np.array([[1., 0.], [0., 1.], [1., 1.]])
ex2_rhs = np.array([[2., 2.], [1., 1.], [3., 0.]])
# TODO: ex2_solutions = ...   # (2, 2)
# TODO: ex2_residual = ...    # (3, 2)
# TODO: ex2_in_range = ...    # (2,), boolean

In [12]:
if ready('ex2_solutions', 'ex2_residual', 'ex2_in_range'):
    assert ex2_solutions.shape == (2, 2) and ex2_residual.shape == ex2_rhs.shape
    np.testing.assert_allclose(ex2_residual, ex2_rhs - ex2_A @ ex2_solutions, atol=ATOL)
    np.testing.assert_allclose(ex2_A.T @ ex2_residual, 0., atol=ATOL)
    np.testing.assert_allclose(ex2_solutions[:, 0], [2., 1.], atol=ATOL)
    np.testing.assert_array_equal(ex2_in_range, [True, False])
    print('Bài 2 đạt.')

Cần điền TODO cho ex2_solutions, ex2_residual, ex2_in_range


<details>
<summary>Đáp án Bài 2</summary>

```python
ex2_solutions = np.linalg.lstsq(ex2_A, ex2_rhs, rcond=None)[0]
ex2_residual = ex2_rhs - ex2_A @ ex2_solutions
ex2_in_range = np.linalg.norm(ex2_residual, axis=0) <= ATOL
```

</details>

### Bài 3 — Kernel của ma trận rộng và left kernel của ma trận cao

Dùng SVD đầy đủ để tìm basis kernel của `ex3_wide` và left null space của `ex3_tall`. Dùng dạng rút gọn cho ma trận rộng rồi đếm số hướng kernel lấy được.

In [13]:
ex3_wide = np.array([[1., 0., 2., 3., 0.], [0., 1., 0., 4., 1.]])
ex3_tall = ex3_wide.T
# TODO: ex3_wide_null = ...     # (5, 3)
# TODO: ex3_tall_left_null = ... # (5, 3)
# TODO: ex3_reduced_count = ...

In [14]:
if ready('ex3_wide_null', 'ex3_tall_left_null', 'ex3_reduced_count'):
    for basis in (ex3_wide_null, ex3_tall_left_null):
        assert basis.shape == (5, 3)
        np.testing.assert_allclose(basis.T @ basis, np.eye(3), atol=ATOL)
    np.testing.assert_allclose(ex3_wide @ ex3_wide_null, 0., atol=ATOL)
    np.testing.assert_allclose(ex3_tall.T @ ex3_tall_left_null, 0., atol=ATOL)
    assert ex3_reduced_count == 0
    print('Bài 3 đạt.')

Cần điền TODO cho ex3_wide_null, ex3_tall_left_null, ex3_reduced_count


<details>
<summary>Đáp án Bài 3</summary>

```python
ex3_rank = np.linalg.matrix_rank(ex3_wide)
_, _, ex3_full_vt = np.linalg.svd(ex3_wide, full_matrices=True)
ex3_full_u, _, _ = np.linalg.svd(ex3_tall, full_matrices=True)
_, _, ex3_thin_vt = np.linalg.svd(ex3_wide, full_matrices=False)
ex3_wide_null = ex3_full_vt[ex3_rank:].T
ex3_tall_left_null = ex3_full_u[:, ex3_rank:]
ex3_reduced_count = ex3_thin_vt[ex3_rank:].shape[0]
```

Ma trận rộng full row rank vẫn có ba hướng kernel vì có năm biến và hai pivot.

</details>

### Bài 4 — Thay đổi scale và tolerance

Tính numerical rank trước và sau khi nhân cả ma trận với `1e3`. So sánh ngưỡng tuyệt đối `1e-6` với ngưỡng thay đổi theo `1e-6 * s.max()` cho từng ma trận.

In [15]:
ex4_matrices = [np.diag([2., 2e-8]), np.diag([2., 2e-8]) * 1e3]
# TODO: ex4_absolute = ...  # hai rank với cùng ngưỡng tuyệt đối
# TODO: ex4_relative = ...  # hai rank với ngưỡng theo singular value lớn nhất

In [16]:
if ready('ex4_absolute', 'ex4_relative'):
    np.testing.assert_array_equal(ex4_absolute, [1, 2])
    np.testing.assert_array_equal(ex4_relative, [1, 1])
    print('Bài 4 đạt.')

Cần điền TODO cho ex4_absolute, ex4_relative


<details>
<summary>Đáp án Bài 4</summary>

```python
ex4_absolute = []
ex4_relative = []
for matrix in ex4_matrices:
    ex4_s = np.linalg.svd(matrix, compute_uv=False)
    ex4_absolute.append(np.linalg.matrix_rank(matrix, tol=1e-6))
    ex4_relative.append(np.linalg.matrix_rank(matrix, tol=1e-6 * ex4_s.max()))
```

Nhân ma trận với hằng số làm đổi singular value nhưng giữ tỷ lệ giữa chúng.

</details>

### Bài 5 — Hai mô hình trùng dự đoán train, khác trên mẫu mới

Fit `ex5_w` bằng `lstsq`, rồi cộng một vector kernel khác zero để có `ex5_shifted_w`. Tính độ thay đổi dự đoán trên train và trên mẫu mới. Giải thích vai trò của quan hệ giữa hai feature.

In [17]:
ex5_t = np.array([-1., 0., 1., 2.])
ex5_X = np.column_stack([np.ones_like(ex5_t), ex5_t, 3. * ex5_t])
ex5_y = 2. + 5. * ex5_t
ex5_X_new = np.array([[1., 2., 5.]])
# TODO: ex5_w = ...
# TODO: ex5_shifted_w = ...
# TODO: ex5_train_diff = ...
# TODO: ex5_new_diff = ...

In [18]:
if ready('ex5_w', 'ex5_shifted_w', 'ex5_train_diff', 'ex5_new_diff'):
    assert ex5_w.shape == ex5_shifted_w.shape == (3,)
    assert not np.allclose(ex5_w, ex5_shifted_w)
    np.testing.assert_allclose(ex5_X @ ex5_w, ex5_y, atol=ATOL)
    np.testing.assert_allclose(ex5_train_diff, ex5_X @ (ex5_shifted_w - ex5_w), atol=ATOL)
    np.testing.assert_allclose(ex5_train_diff, np.zeros(4), atol=ATOL)
    np.testing.assert_allclose(ex5_new_diff, ex5_X_new @ (ex5_shifted_w - ex5_w), atol=ATOL)
    assert np.linalg.norm(ex5_new_diff) > ATOL
    print('Bài 5 đạt.')

Cần điền TODO cho ex5_w, ex5_shifted_w, ex5_train_diff, ex5_new_diff


<details>
<summary>Đáp án Bài 5</summary>

```python
ex5_w = np.linalg.lstsq(ex5_X, ex5_y, rcond=None)[0]
ex5_shifted_w = ex5_w + np.array([0., 3., -1.])
ex5_train_diff = ex5_X @ (ex5_shifted_w - ex5_w)
ex5_new_diff = ex5_X_new @ (ex5_shifted_w - ex5_w)
```

Feature cuối của mẫu mới bằng 5, trong khi quan hệ trên train cho $3\cdot2=6$.

</details>

### Bài 6 — Nén bảng điểm có nhiễu

Thêm nhiễu có seed 42 vào bảng điểm ở phần 6. Tái tạo với `k = 1, 2, 3`, tính rank và sai số Frobenius của từng bản tái tạo. So sánh sai số với norm của phần singular value bị bỏ.

In [19]:
ex6_ratings = ratings + np.random.default_rng(42).normal(scale=0.05, size=ratings.shape)
ex6_left, ex6_s, ex6_vt = np.linalg.svd(ex6_ratings, full_matrices=False)
ex6_ks = (1, 2, 3)
# TODO: ex6_reconstructed = ...  # ba ma trận tái tạo theo thứ tự k
# TODO: ex6_ranks = ...
# TODO: ex6_errors = ...

In [20]:
if ready('ex6_reconstructed', 'ex6_ranks', 'ex6_errors'):
    assert np.shape(ex6_reconstructed) == (3, *ratings.shape)
    actual_ranks = [np.linalg.matrix_rank(matrix) for matrix in ex6_reconstructed]
    assert np.linalg.matrix_rank(ex6_ratings) == 4
    assert actual_ranks == list(ex6_ks)
    np.testing.assert_array_equal(ex6_ranks, actual_ranks)
    actual_errors = [np.linalg.norm(ex6_ratings - matrix, ord='fro') for matrix in ex6_reconstructed]
    np.testing.assert_allclose(ex6_errors, actual_errors, atol=ATOL)
    np.testing.assert_allclose(ex6_errors, [np.linalg.norm(ex6_s[k:]) for k in ex6_ks], atol=ATOL)
    assert ex6_errors[0] > ex6_errors[1] > ex6_errors[2] > 0.
    print('Bài 6 đạt.')

Cần điền TODO cho ex6_reconstructed, ex6_ranks, ex6_errors


<details>
<summary>Đáp án Bài 6</summary>

```python
ex6_reconstructed = [(ex6_left[:, :k] * ex6_s[:k]) @ ex6_vt[:k] for k in ex6_ks]
ex6_ranks = [np.linalg.matrix_rank(matrix) for matrix in ex6_reconstructed]
ex6_errors = [np.linalg.norm(ex6_ratings - matrix, ord='fro') for matrix in ex6_reconstructed]
```

Nhiễu thêm các hướng nhỏ. Giữ hai hướng đầu vẫn cho sai số thấp hơn nhiều so với chỉ giữ một hướng trong dữ liệu này.

</details>

### Bài 7 — Quan hệ giữa các cột phim

Lập quan hệ `Hành động + Hoạt hình = 2 * Phiêu lưu` thành vector `ex7_relation`. Lấy cơ sở trực chuẩn của kernel từ SVD đầy đủ, rồi biểu diễn vector quan hệ bằng cơ sở đó.

In [21]:
# TODO: ex7_relation = ...    # (5,), một hệ số trên mỗi phim
# TODO: ex7_basis = ...       # (5, 3), theo thứ tự cột
# TODO: ex7_rebuilt = ...     # (5,), tái tạo từ tọa độ trong basis

In [22]:
if ready('ex7_relation', 'ex7_basis', 'ex7_rebuilt'):
    assert ex7_relation.shape == (5,) and ex7_basis.shape == (5, 3)
    assert ex7_relation[0] != 0.
    np.testing.assert_allclose(ex7_relation / ex7_relation[0], [1., 1., -2., 0., 0.], atol=ATOL)
    np.testing.assert_allclose(ratings @ ex7_basis, 0., atol=ATOL)
    np.testing.assert_allclose(ex7_basis.T @ ex7_basis, np.eye(3), atol=ATOL)
    np.testing.assert_allclose(ex7_rebuilt, ex7_basis @ (ex7_basis.T @ ex7_relation), atol=ATOL)
    np.testing.assert_allclose(ex7_rebuilt, ex7_relation, atol=ATOL)
    print('Bài 7 đạt.')

Cần điền TODO cho ex7_relation, ex7_basis, ex7_rebuilt


<details>
<summary>Đáp án Bài 7</summary>

```python
ex7_relation = np.array([1., 1., -2., 0., 0.])
_, _, ex7_full_vt = np.linalg.svd(ratings, full_matrices=True)
ex7_basis = ex7_full_vt[np.linalg.matrix_rank(ratings):].T
ex7_rebuilt = ex7_basis @ (ex7_basis.T @ ex7_relation)
```

</details>

## Bài tổng hợp — Gợi ý phim cho người dùng mới

Học hai hướng phim từ `ratings` ở phần 6, bảng đầy đủ của bốn **người dùng cũ**.
Người dùng mới đã chấm hai phim ở chỉ số `[0, 2]`. Điểm phim chỉ số `3` được giữ
riêng để kiểm tra và chỉ dùng sau khi đã tạo dự đoán.

1. Dùng SVD cắt ngọn để lấy `cap_movies` shape `(2, 5)` từ bảng người dùng cũ.
2. Tạo hệ `cap_design` có một hàng cho mỗi phim đã chấm. Fit vector `cap_user`
   bằng `lstsq`, lưu `cap_rank`, rồi dự đoán điểm cho cả năm phim.
3. Tính sai số tuyệt đối trên điểm giữ lại. Loại hai phim đã dùng để fit
   khỏi tập ứng viên và chọn một phim có điểm dự đoán cao nhất.
4. Chỉ giữ phim đã chấm đầu tiên. Fit lại và lấy một hướng kernel đơn vị từ
   SVD đầy đủ. Cộng hướng đó vào vector người dùng. Đo thay đổi điểm trên
   phim đã dùng và trên các phim còn lại.

### Bài tổng hợp — Fit, đánh giá và kiểm tra kernel

Dự đoán rank của hai hệ trước khi chạy. Dùng mask riêng để lưu phim đã biết.

In [23]:
cap_k = 2
cap_rated = np.array([0, 2])
cap_observed = np.array([3., 2.])
cap_holdout_id = 3
cap_holdout_score = 3.5
# TODO: cap_movies = ...       # (2, 5), hai hàng đầu của Vt từ ratings
# TODO: cap_design = ...       # (2, 2), hai hàng tương ứng với cap_rated
# TODO: cap_user, _, cap_rank, _ = ...
# TODO: cap_predictions = ...  # (5,)
# TODO: cap_holdout_error = ...
# TODO: cap_watched = ...      # (5,), True trên hai phim đã dùng để fit
# TODO: cap_top_movie = ...    # tên phim
# TODO: cap_one_design = ...   # (1, 2), chỉ giữ hàng đầu tiên
# TODO: cap_one_user, _, cap_one_rank, _ = ...
# TODO: cap_null_direction = ...  # (2,), hướng đơn vị trong kernel
# TODO: cap_score_change = ...    # (5,), hiệu dự đoán sau và trước khi dịch

In [24]:
if ready('cap_movies', 'cap_design', 'cap_user', 'cap_rank', 'cap_predictions', 'cap_holdout_error', 'cap_watched', 'cap_top_movie', 'cap_one_design', 'cap_one_user', 'cap_one_rank', 'cap_null_direction', 'cap_score_change'):
    assert cap_movies.shape == (cap_k, ratings.shape[1])
    np.testing.assert_allclose(cap_movies @ cap_movies.T, np.eye(cap_k), atol=ATOL)
    np.testing.assert_allclose(ratings @ cap_movies.T @ cap_movies, ratings, atol=ATOL)
    np.testing.assert_allclose(cap_design, cap_movies[:, cap_rated].T, atol=ATOL)
    assert cap_user.shape == (cap_k,) and cap_rank == 2
    np.testing.assert_allclose(cap_design @ cap_user, cap_observed, atol=ATOL)
    np.testing.assert_allclose(cap_predictions, cap_user @ cap_movies, atol=ATOL)
    np.testing.assert_allclose(cap_predictions, [3., 1., 2., 3.5, 2.5], atol=ATOL)
    np.testing.assert_allclose(cap_holdout_error, 0., atol=ATOL)
    np.testing.assert_array_equal(cap_watched, [True, False, True, False, False])
    assert cap_top_movie == 'Trinh thám'
    np.testing.assert_allclose(cap_one_design, cap_design[:1], atol=ATOL)
    assert cap_one_rank == 1 and cap_one_user.shape == cap_null_direction.shape == (2,)
    np.testing.assert_allclose(cap_one_design @ cap_one_user, cap_observed[:1], atol=ATOL)
    np.testing.assert_allclose(np.linalg.norm(cap_null_direction), 1., atol=ATOL)
    np.testing.assert_allclose(cap_one_design @ cap_null_direction, 0., atol=ATOL)
    np.testing.assert_allclose(cap_score_change, cap_null_direction @ cap_movies, atol=ATOL)
    np.testing.assert_allclose(cap_score_change[cap_rated[0]], 0., atol=ATOL)
    assert np.linalg.norm(np.delete(cap_score_change, cap_rated[0])) > ATOL
    print('Bài tổng hợp đạt. Phim gợi ý:', cap_top_movie)

Cần điền TODO cho cap_movies, cap_design, cap_user, cap_rank, cap_predictions, cap_holdout_error, cap_watched, cap_top_movie, cap_one_design, cap_one_user, cap_one_rank, cap_null_direction, cap_score_change


<details>
<summary>Đáp án bài tổng hợp</summary>

```python
_, _, cap_vt = np.linalg.svd(ratings, full_matrices=False)
cap_movies = cap_vt[:cap_k]
cap_design = cap_movies[:, cap_rated].T
cap_user, _, cap_rank, _ = np.linalg.lstsq(cap_design, cap_observed, rcond=None)
cap_predictions = cap_user @ cap_movies
cap_holdout_error = abs(cap_predictions[cap_holdout_id] - cap_holdout_score)

cap_watched = np.zeros(ratings.shape[1], dtype=bool)
cap_watched[cap_rated] = True
cap_candidates = np.where(~cap_watched, cap_predictions, -np.inf)
cap_top_movie = movies[np.argmax(cap_candidates)]

cap_one_design = cap_design[:1]
cap_one_user, _, cap_one_rank, _ = np.linalg.lstsq(cap_one_design, cap_observed[:1], rcond=None)
_, _, cap_one_vt = np.linalg.svd(cap_one_design, full_matrices=True)
cap_null_direction = cap_one_vt[cap_one_rank]
cap_score_change = (cap_one_user + cap_null_direction) @ cap_movies - cap_one_user @ cap_movies
```

Hai hàng độc lập xác định đủ hai tọa độ người dùng trong ví dụ này. Khi chỉ giữ một hàng, chiều tự do còn lại cho các dự đoán khác nhau trên phim chưa chấm. Sai số gần zero trên điểm giữ lại đến từ cách dựng dữ liệu hạng 2.

</details>

## Tự kiểm tra nhanh

1. Với `A.shape == (4, 7)` và rank 3, basis của kernel và left null space có shape gì?
2. Vì sao `lstsq` có thể trả về trọng số ngay cả khi $Ax=b$ không có nghiệm chính xác?
3. Hai basis kernel tìm từ hai lần SVD có vector khác dấu. Bạn kiểm tra chúng
   sinh cùng không gian bằng cách nào?
4. Thử so sánh ma trận `(2, 2)` hạng 1 và ma trận `(1, 2)` hạng 1 khi fit người
   dùng mới. Vì sao cả hai đều để lại một hướng chưa xác định?
5. Nếu sai số tái tạo giảm khi tăng $k$, bạn cần thêm dữ liệu nào để biết điểm
   dự đoán cho phim chưa chấm có chính xác hơn không?

## Tổng kết

- $\operatorname{Col}(A)$ chứa mọi đầu ra, còn $\ker(A)$ chứa các hướng bị triệt tiêu.
- Rank là số hướng độc lập ở cả hai phía. Nullity bằng số cột trừ rank.
- SVD đầy đủ cho đủ basis kernel ở hai phía. Dùng shape, phần dư và
  tính trực chuẩn để kiểm tra.
- Khi singular value nhỏ, numerical rank thay đổi theo tolerance và scale.
- Trong hồi quy, các hướng kernel tạo ra các trọng số khác nhau với cùng dự đoán
  trên design matrix đang xét. Trong hệ gợi ý, rank giới hạn số hướng điểm
  có thể biểu diễn và ảnh hưởng đến việc fit người dùng mới.

Trước khi sang Bài 07, hãy giải thích vì sao phần dư least squares vuông góc với
column space dù phần dư có thể khác zero.